# M2a 일반화와 과적합 — 실습 (W3, M2 2부작 1편)

> ⚠️ **가장 먼저 — 화면 위 [Drive로 복사]를 누르세요.**
> 지금 보고 있는 것은 원본을 잠깐 띄운 **임시 사본**입니다. 복사하지 않으면 탭을 닫는 순간
> 채운 빈칸과 실행 결과가 **모두 사라집니다.** 복사본은 내 Google Drive에 저장되고,
> 원본은 바뀌지 않으니 마음껏 고쳐도 됩니다.

> 위에서부터 한 셀씩 `Shift+Enter`로 실행하세요. `___` 빈칸은 직접 채웁니다.

**이 실습이 끝나면**
1. train/test 분할을 하고 **stratify 손계산**([10, 10, 10])을 검증한다
2. 트리 깊이를 키우며 **과적합(train 1.000 vs test 0.918)을 눈으로** 본다
3. **누수의 함정** ⭐ — 무작위 데이터에서 "가짜 91%"를 직접 만들어 보고, 올바른 순서(53%)와 대조한다

**7단계 멘탈모델 초점:** 평가(Evaluation)

## Part A. 공정한 시험 만들기 — 분할과 stratify
붓꽃 150송이 = 세 품종 50/50/50. test 20%면 시험지는 30송이 — stratify=y라면 **[10, 10, 10]** 이어야 합니다(먼저 종이에서 확인).

In [ ]:
import numpy as np                                    # 수치 계산
import matplotlib.pyplot as plt                       # 그래프
from sklearn.datasets import load_iris                # 붓꽃 데이터
from sklearn.model_selection import train_test_split  # 학습/시험 분리

iris = load_iris()                                    # 150송이, 3품종(50/50/50)
X, y = iris.data, iris.target                         # X=특징 4개, y=품종(0/1/2)

X_train, X_test, y_train, y_test = train_test_split(  # 공정한 시험 만들기
    X, y, test_size=___,                              # ✍️ 빈칸: 20%를 시험지로
    random_state=42, stratify=___)                    # ✍️ 빈칸: 클래스 비율을 유지할 기준(정답 배열)
print('train:', X_train.shape, '| test:', X_test.shape)
print('시험셋 품종 분포(stratify O):', np.bincount(y_test))   # [10 10 10] — 손계산 그대로?

_, _, _, y_test_no = train_test_split(X, y, test_size=0.2, random_state=7)  # stratify 없이
print('시험셋 품종 분포(stratify X):', np.bincount(y_test_no))  # 쏠림!

> **검산 포인트:** stratify=y → **[10, 10, 10]**(손계산 일치). 없으면 [7, 12, 11]처럼 운에 따라 쏠림 — 어떤 품종은 7문제만 출제되는 **불공정한 시험**이 됩니다. `random_state`는 재현성(누구나 같은 분할).

## Part B. 과적합을 눈으로 — 복잡도 스윕
유방암 진단 데이터에서 결정트리의 깊이(복잡도 손잡이)를 1→15로 키우며 **두 점수를 나란히** 봅니다. (결정트리 자체는 M6에서 — 지금은 "복잡도 조절이 되는 모델"이면 충분)

In [ ]:
from sklearn.datasets import load_breast_cancer       # 유방암 데이터(이진분류)
from sklearn.tree import DecisionTreeClassifier        # 복잡도 손잡이(max_depth) 달린 모델

data = load_breast_cancer()                            # 569명, 특징 30개
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(   # 30% 시험, 비율 유지
    X, y, test_size=0.3, random_state=42, stratify=y)

depths = range(1, 16)                                  # 깊이 1~15
train_acc, test_acc = [], []
for d in depths:                                       # 복잡도를 키워가며
    m = DecisionTreeClassifier(max_depth=___, random_state=0)  # ✍️ 빈칸: 이번 반복의 깊이
    m.fit(X_train, y_train)                            # train으로 학습
    train_acc.append(m.score(___, y_train))            # ✍️ 빈칸: 학습 데이터 점수(무엇으로 채점?)
    test_acc.append(m.score(X_test, ___))              # ✍️ 빈칸: 시험 데이터 점수(정답은?)

plt.plot(depths, train_acc, 'o-', label='train')       # 두 곡선을 나란히
plt.plot(depths, test_acc, 's-', label='test')
plt.xlabel('max_depth (model complexity)'); plt.ylabel('accuracy')  # 축(영어)
plt.legend(); plt.grid(True, alpha=0.3)
plt.title('Overfitting as the tree gets deeper')
plt.show()
print('depth 3 :', round(train_acc[2], 3), '/', round(test_acc[2], 3))   # 0.980 / 0.924(최고)
print('depth 15:', round(train_acc[-1], 3), '/', round(test_acc[-1], 3)) # 1.000 / 0.918

> **관찰:** train은 깊이 15에서 **만점(1.000)** — 그런데 test는 0.918로 **깊이 3(0.924)보다 낮습니다.** 격차 0.082 = 과적합. **train 만점은 자랑이 아니라 경고**이고, 최고의 시험 점수는 "적당한 복잡도"에서 나옵니다. (2학기 신경망의 학습 곡선에서 같은 진단법이 재등장 — 2학기 D1)

## Part C. 누수의 함정 — 가짜 91% 만들기 ⭐
표본 100개 × 특징 1,000개를 **전부 난수**로, 정답 y도 동전 던지기로 만듭니다. 특징과 정답이 무관하므로 **진짜 실력은 50%**. 그런데 "특징 선택"의 순서를 틀리면…

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif  # 특징 선택 도구
from sklearn.linear_model import LogisticRegression            # 간단한 분류기

rng = np.random.default_rng(0)                         # 재현성
Xr = rng.normal(size=(100, 1000))                      # 특징 1,000개 — 전부 난수!
yr = rng.integers(0, 2, 100)                           # 정답 — 동전 던지기!

sel = SelectKBest(f_classif, k=20).fit(Xr, yr)         # ❌ 전체 데이터로 "좋아 보이는" 20개 선택
Xr_sel = sel.transform(Xr)                             # (시험 정보까지 보고 골랐다 = 누수)

accs_wrong = []
for rs in range(10):                                   # 분할 10가지로 평균(운 배제)
    Xtr, Xte, ytr, yte = train_test_split(Xr_sel, yr, test_size=0.3, random_state=rs, stratify=yr)
    lr = LogisticRegression(max_iter=5000).fit(Xtr, ytr)
    accs_wrong.append(lr.score(Xte, yte))              # 시험 점수(가짜!)
print('❌ 잘못된 순서(선택→분할) 평균 정확도:', round(np.mean(accs_wrong), 3))  # ~0.91?!

In [ ]:
accs_right = []
for rs in range(10):                                   # 같은 데이터, 순서만 바로잡기
    Xtr2, Xte2, ytr2, yte2 = train_test_split(Xr, yr, test_size=0.3, random_state=rs, stratify=yr)
    sel2 = SelectKBest(f_classif, k=20).fit(___, ytr2) # ✍️ 빈칸: 선택은 무엇만 보고? (train 특징)
    lr2 = LogisticRegression(max_iter=5000).fit(sel2.transform(Xtr2), ytr2)
    accs_right.append(lr2.score(sel2.transform(Xte2), yte2))
print('✅ 올바른 순서(분할→train만 선택) 평균 정확도:', round(np.mean(accs_right), 3))  # ~0.53

plt.bar([0, 1], [np.mean(accs_wrong), np.mean(accs_right)],  # 두 순서 비교
        color=['#dc2626', '#2563eb'], width=0.5,
        yerr=[np.std(accs_wrong), np.std(accs_right)], capsize=6)
plt.xticks([0, 1], ['leaky (select before split)', 'correct (select after split)'])
plt.ylabel('test accuracy (10 splits)')                # 축(영어)
plt.axhline(0.5, color='gray', ls='--', lw=1)          # 진짜 실력 = 0.5
plt.title('Random data: leakage fakes a good score')
plt.tight_layout(); plt.show()

> **가짜 91% vs 정직한 53%.** 난수 특징 1,000개 중 우연히 y와 닮은 놈이 있는데, "전체로 고르기"는 **시험지까지 보고 그 우연을 커닝**한 것. 모델이 아니라 **절차가 부정행위**를 했습니다. 원칙: **분할 먼저, train만으로**(스케일링·결측 대치·특징 선택 전부). "너무 좋은 점수"는 기뻐하기 전에 누수부터 의심 — 프로의 습관.

## 진단 연습 — 4명의 환자

| | train | test | 진단 |
|---|---|---|---|
| A | 0.62 | 0.60 | ? |
| B | 0.99 | 0.71 | ? |
| C | 0.95 | 0.93 | ? |
| D | 0.99 | 0.99 | ? |

<details><summary>정답 보기</summary>
A=과소적합(둘 다 낮음 — 모델을 키워라) · B=과적합(격차 — 단순화·데이터↑) · C=적합 · <b>D=너무 좋다 — 누수부터 의심</b>(절차 점검이 첫 반응).
</details>

## 🤖 AI 코파일럿 활용 (선택) — ai-native v1
막히면 AI 튜터에게 묻되, **먼저 스스로 생각**하고 답을 **실행으로 검증**하세요.

**좋은 질문 예시**
- "붓꽃 150송이·test 20%·stratify의 시험셋 분포를 내가 계산해 볼 테니 채점해 줘."
- "train 1.000 / test 0.918 — 이 모델의 상태 진단과 처방을 내가 말해 볼게."
- "무작위 데이터에서 91%가 나온 이유를 '커닝'으로 설명해 볼게 — 허점을 찔러 줘."
- "k(선택 특징 수)를 20에서 5나 100으로 바꾸면 가짜 점수가 어떻게 변할지 예측해 볼게 — 실행으로 검증할게."

**가드레일**
1. 먼저 손으로 생각 → 그 다음 AI
2. AI 코드는 *왜 그런지* 설명할 수 있을 때만 사용
3. AI 출력은 실행으로 검증

## 정리 & 자가 점검

**오늘 한 일 3줄**
1. train/test 분할과 stratify를 손계산([10,10,10])으로 검증했다
2. 깊이 스윕으로 과적합(1.000 vs 0.918, 격차 0.082)을 눈으로 봤다 — train 만점은 경고
3. 무작위 데이터에서 **가짜 91%**(누수)를 만들고, 올바른 순서로 53%(정직)를 확인했다

**스스로 점검**
- [ ] "공정한 시험"이 왜 필요한지 한 문장으로 말할 수 있다
- [ ] train·test 점수 조합 4가지를 진단할 수 있다(D는 누수 의심!)
- [ ] "분할 먼저, train만으로" 원칙이 적용되는 3가지 전처리를 안다
- [ ] 너무 좋은 점수를 봤을 때의 첫 반응을 안다

**🔹심화 (선택)**
- **k 실험:** SelectKBest의 k를 5/20/100으로 바꿔 가짜 점수의 변화를 관찰하세요(우연 후보가 많을수록?).
- **표본 수 실험:** n=100을 1,000으로 늘리면 가짜 점수가 어떻게 변할까요(우연이 살아남기 어려워짐).
- **학습 곡선:** `sklearn.model_selection.learning_curve`로 "데이터를 더 모으면 나아질까"를 그려 보세요.

**다음 시간(M2b):** 점수를 올바르게 읽는 법 — 혼동행렬 손계산·정확도의 함정·교차검증.